# Polygon Type Bayesian Setup

This notebook extends `polygons_type_setup.ipynb` with a **cluster-weighted prior**
derived from `Population_Cluster_Proposed.csv`, replacing the earlier `Prior.csv`
TM-weighted-by-county mixture.

**Key change**: for each polygon, if its `Clusters` column names a proposed population
cluster (e.g. "Westminster"), the prior is that cluster's own distribution over type
codes — looked up by `(Region, Location, Cluster_Name)` in
`csv_input/Info_CSV/Population_Cluster_Proposed.csv`, restricted to the valid types for
that (Region, Location) and renormalised. Polygons with no cluster, an unmatched cluster,
or a cluster whose restricted mixture sums to 0 fall back to a flat uniform prior over
the valid types.

**Scope of this pass**: only blank-`County_Type` polygons that *also* have a non-blank
`Clusters` value are classified here. Polygons without a `Clusters` value are left blank
and neglected for now — this stage doesn't attempt to guess a type for them.

$$\pi(k \mid r, \ell, c) =
\begin{cases}
  P_{\text{cluster}}(k \mid r, \ell, c) & \text{cluster present \& matched}, k \in \mathcal{V}(r,\ell) \\[4pt]
  \dfrac{1}{|\mathcal{V}(r,\ell)|} & \text{otherwise}
\end{cases}$$

**Stage 5 polygon data** (`07242119_modified_shapefile.csv`, 20,850 rows) is split into 5
location-type DataFrames matching the community taxonomy defined in
`csv_input/Info_CSV/Community_matrix.csv`. Its `Region` values (`Alma Valley East/West`,
`Midlands`, `Highland`, `South East`) are remapped to the `Capital`/`Highland`/`Lowland`
taxonomy that `Community_matrix.csv` and `Population_Cluster_Proposed.csv` use, in the
cell right after loading.

| Variable | Location Type | Transition Matrix File |
|---|---|---|
| `df_inner_city` | Inner City | `Capital_InnerCity.csv` |
| `df_outer_city` | Outer City | `Capital_OuterCity.csv` |
| `df_rural` | Rural | `Capital_Rural.csv` |
| `df_suburb` | Suburb | `Capital_Suburb.csv` |
| `df_town` | Town | `Capital_Town.csv` |

In [88]:
import sys, os
import pandas as pd
from shapely import wkt
from shapely.strtree import STRtree

# Import random_no_generation as a proper module so CSV_FILE stays in its own namespace
sys.path.insert(0, os.path.abspath('..'))
import random_no_generation as randomnogen

# Patch CSV_FILE to the correct absolute path (module lives one level up, notebook does not)
randomnogen.CSV_FILE = os.path.abspath('../randomno.csv')
print(f'CSV_FILE : {randomnogen.CSV_FILE}')
print(f'Exists   : {os.path.exists(randomnogen.CSV_FILE)}')

# Alias functions into the notebook namespace so the rest of the code is unchanged
return_ran                     = randomnogen.return_ran
return_ran_array               = randomnogen.return_ran_array
random_normal                  = randomnogen.random_normal
random_normal_from_variance    = randomnogen.random_normal_from_variance
random_normal_adv              = randomnogen.random_normal_adv
random_normal_adv_from_variance = randomnogen.random_normal_adv_from_variance
clean_index                    = randomnogen.clean_index

# Smoke-test
print(f'Test random int: {return_ran()}')

CSV_FILE : e:\Coding Site\just_for_fun\randomno.csv
Exists   : True
Test random int: 6470477


In [89]:
# Load main stage 5 polygon dataset
target_df = pd.read_csv('./csv_input/07280017_modified_shapefile.csv')

print(f'Total polygons: {len(target_df)}')
print(f'Columns: {list(target_df.columns)}')
print('\nLocation type counts:')
print(target_df['Location'].value_counts())

Total polygons: 20850
Columns: ['Zone', 'County', 'District', 'Shape_ID', 'geometry', 'Area', 'Population_Density', 'Region', 'Location', 'County_Type', 'Capacity', 'Population', 'District_Ward', 'Clusters', 'Unnamed: 14']

Location type counts:
Location
Inner City    9789
Outer City    7537
Suburb        1590
Rural         1428
Town           506
Name: count, dtype: int64


In [90]:
# ── Remap Region to the taxonomy known by Community_matrix.csv / Population_Cluster_Proposed.csv ──
# The source file's Region values (Alma Valley East/West, Midlands, Highland,
# South East) come from a finer real-world geography than these reference files
# currently model (Capital/Highland/Lowland). Any Region not in that taxonomy has
# no entry in valid_types_map or cluster_priors, so every blank County_Type row
# under it would silently fail to be assigned (no error raised).
REGION_MAP = {
    'Alma Valley East': 'Capital',
    'Alma Valley West': 'Capital',
    'South East'      : 'Lowland',
    'Midlands'        : 'Lowland',
    # 'Highland' already matches — no remap needed
}

print('Region counts before remap:')
print(target_df['Region'].value_counts())

target_df['Region'] = target_df['Region'].replace(REGION_MAP)

print('\nRegion counts after remap:')
print(target_df['Region'].value_counts())

Region counts before remap:
Region
Alma Valley East    8955
Midlands            5716
Highland            3737
Alma Valley West    1709
South East           733
Name: count, dtype: int64

Region counts after remap:
Region
Capital     10664
Lowland      6449
Highland     3737
Name: count, dtype: int64


In [91]:
# Split into 5 sub-DataFrames by location type
# Names match the community taxonomy in Community_matrix.csv

df_inner_city = target_df[target_df['Location'] == 'Inner City'].reset_index(drop=True)
df_outer_city = target_df[target_df['Location'] == 'Outer City'].reset_index(drop=True)
df_rural      = target_df[target_df['Location'] == 'Rural'].reset_index(drop=True)
df_suburb     = target_df[target_df['Location'] == 'Suburb'].reset_index(drop=True)
df_town       = target_df[target_df['Location'] == 'Town'].reset_index(drop=True)

print('Polygon counts by location type:')
print(f'  Inner City : {len(df_inner_city)}')
print(f'  Outer City : {len(df_outer_city)}')
print(f'  Rural      : {len(df_rural)}')
print(f'  Suburb     : {len(df_suburb)}')
print(f'  Town       : {len(df_town)}')
total = (len(df_inner_city) + len(df_outer_city) + len(df_rural)
         + len(df_suburb) + len(df_town))
print(f'  Total      : {total}')

Polygon counts by location type:
  Inner City : 9789
  Outer City : 7537
  Rural      : 1428
  Suburb     : 1590
  Town       : 506
  Total      : 20850


In [92]:
# Load all Transition Matrix files
# Keys match the Location type names from Community_matrix.csv

TRANSITION_MATRIX_DIR = './csv_input/Info_CSV/Transition_Matrix'

transition_matrix_files = {
    'Inner City': 'Capital_InnerCity.csv',
    'Outer City': 'Capital_OuterCity.csv',
    'Rural'     : 'Capital_Rural.csv',
    'Suburb'    : 'Capital_Suburb.csv',
    'Town'      : 'Capital_Town.csv',
}

transition_matrices = {
    location: pd.read_csv(
        os.path.join(TRANSITION_MATRIX_DIR, filename),
        index_col='FROM_TO'
    )
    for location, filename in transition_matrix_files.items()
}

print('Transition matrices loaded:')
for location, tm in transition_matrices.items():
    print(f'  {location:12s}: {tm.shape[0]} x {tm.shape[1]}  columns={list(tm.columns)}')

Transition matrices loaded:
  Inner City  : 15 x 15  columns=['S', 'A1', 'A2', 'B1', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2', 'TECH', 'GAME', 'UNI']
  Outer City  : 12 x 12  columns=['A1', 'A2', 'B1', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Rural       : 5 x 5  columns=['A1', 'C1', 'D1', 'D2', 'F1']
  Suburb      : 7 x 7  columns=['A1', 'B1', 'C1', 'C2', 'E2', 'F1', 'UNI']
  Town        : 6 x 6  columns=['B2', 'C1', 'C2', 'D2', 'E1', 'F1']


In [93]:
# Load Population_Cluster_Proposed.csv — replaces Prior.csv as the prior source.
# Columns : Region, Type, Key, Cluster_Name, Description, <type codes...>
# Values  : percentage chance a polygon in this (Region, Type, Cluster_Name) is that type.
# A polygon opts into a cluster via its own 'Clusters' column in the stage-5 data.
# Rows should sum to 100 — we check but do not force-normalise (a mismatch here
# usually means a data-entry mistake worth looking at, not a units bug).

CLUSTER_CSV_PATH = './csv_input/Info_CSV/Population_Cluster_Proposed.csv'
cluster_df = pd.read_csv(CLUSTER_CSV_PATH)

_meta_cols = {'Region', 'Type', 'Key', 'Cluster_Name', 'Description'}
_type_cols = [c for c in cluster_df.columns if c not in _meta_cols]

row_sums = cluster_df[_type_cols].sum(axis=1)
not_normalised = row_sums[abs(row_sums - 100) > 1e-6]
if len(not_normalised):
    print(f'WARNING: {len(not_normalised)} cluster rows not summing to 100:')
    print(cluster_df.loc[not_normalised.index, ['Region', 'Type', 'Cluster_Name']].to_string())
else:
    print('All cluster rows sum to 100 — no normalisation needed.')

print(f'\nClusters loaded : {len(cluster_df)} rows across {cluster_df["Cluster_Name"].nunique()} distinct names')
print(f'Type columns    : {_type_cols}')
print(f'\nPopulation_Cluster_Proposed.csv preview:')
print(cluster_df[['Region', 'Type', 'Cluster_Name'] + _type_cols].to_string())

All cluster rows sum to 100 — no normalisation needed.

Clusters loaded : 103 rows across 19 distinct names
Type columns    : ['S', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2', 'TECH', 'GAME', 'UNI']

Population_Cluster_Proposed.csv preview:
       Region        Type           Cluster_Name   S  A1  A2  B1  B2  C1  C2  D1  D2  E1  E2  F1  F2  TECH  GAME  UNI
0     Capital  Inner City            Westminster  45  18   6   7   0   7   6   2   1   2   2   2   2     0     0    0
1     Capital  Inner City                   Posh   6  45   6  10   0  10   8   5   2   2   2   2   2     0     0    0
2     Capital  Inner City           Professional   2   4   2  58   0   8   8   5   6   2   2   2   1     0     0    0
3     Capital  Inner City           Middle_Class   1   2   2   8   0  34  29  10   3   4   2   3   2     0     0    0
4     Capital  Inner City                Average   3   5   5   7   0  10  10  22   8   8   8   8   6     0     0    0
5     Capital  Inner Ci

## County_Type Assignment Algorithm (Cluster-Weighted Prior)

Fills `County_Type` for polygons that (a) are currently blank **and** (b) have a
`Clusters` value — this stage only classifies what the cluster data actually covers.
Blank polygons with no `Clusters` value are left untouched for now.

**Steps per iteration:**

1. `unchecked_list` — Shape_IDs with empty/NA `County_Type` **and** a non-blank `Clusters` value
2. Draw `return_ran()` → pick one ID from `unchecked_list`
3. **Prior (cluster-weighted)** — for a polygon with region `r`, location `ℓ`, cluster `c`:
   ```
   π(k | r, ℓ, c) = Population_Cluster_Proposed.csv row (r, ℓ, c), restricted to k ∈ valid_types(r, ℓ)
   ```
   Falls back to uniform over valid types if the cluster has no matching row for (r, ℓ),
   or the restricted mixture sums to 0.
4. Inspect neighbours via the adjacency graph; collect those already classified
5. **Likelihood** — average neighbour TM rows (GAME column excluded — see
   `EXCLUDED_NEIGHBOUR_TYPES`), then blend with `INFLUENCED_WEIGHT` (out of 100):
   - **One-pass seed**: `INFLUENCED_WEIGHT_SEED = 0` — pure prior, neighbours ignored.
     The neighbourhood hasn't settled yet at this point, so seeding is deterministic
     from the cluster prior alone.
   - **Gibbs sweep 1**: `INFLUENCED_WEIGHT_START = 30` (30% neighbour / 70% prior)
   - **Gibbs final sweep**: `INFLUENCED_WEIGHT_END = 50` (50% neighbour / 50% prior),
     ramped linearly across sweeps 1→N_MAX_SWEEPS.
6. Sample type from likelihood. If not valid for location, walk fallback chain:
   `S → A1 → A2 → UNI → TECH → B1 → B2 → C1 → C2 → D2 → D1 → E1 → E2 → F1 → F2`
7. Assign; remove from `unchecked_list`; repeat until empty

**Gibbs sweep note**: capped at 3 sweeps (`N_MAX_SWEEPS`), not run to formal MCMC
convergence — there's no guaranteed stationary distribution here. An earlier version
ramped neighbour weight up to 80% over 5 sweeps, but every candidate is resampled using
other candidates' still-changing values each sweep, so a high, rising neighbour weight
let changes cascade instead of damping out (observed change_rate stayed ~83-87% for all
5 sweeps, never converging). Capping the ceiling at 50% and cutting to 3 sweeps keeps
that feedback loop weaker.

In [94]:
# Create working copy — target_df stays pristine; all writes go to work_df
work_df = target_df.copy()

# Normalise County_Type: treat empty strings as NaN for consistent checks
work_df['County_Type'] = work_df['County_Type'].replace('', pd.NA)

print(f'Total polygons : {len(work_df)}')
print(f'Already typed  : {work_df["County_Type"].notna().sum()}')
print(f'To assign      : {work_df["County_Type"].isna().sum()}')
print('\nPre-assigned type counts:')
print(work_df['County_Type'].value_counts())

Total polygons : 20850
Already typed  : 14941
To assign      : 5909

Pre-assigned type counts:
County_Type
D1      1998
D2      1998
C1      1566
E2      1371
F1      1364
B1      1355
C2      1194
F2      1149
E1      1027
A2       536
A1       492
TECH     485
UNI      195
GAME     109
S         78
B2        24
Name: count, dtype: int64


In [95]:
# Build adjacency graph from polygon geometries
# Two polygons are adjacent if they share a boundary (touches = shared edge/point,
# no interior overlap). Uses a spatial R-tree index for efficiency.

print('Parsing geometries...')
geometries = target_df['geometry'].apply(wkt.loads).tolist()
shape_ids  = target_df['Shape_ID'].tolist()

print('Building spatial index...')
tree = STRtree(geometries)

print('Finding neighbours...')
adjacency = {}
for i, (sid, geom) in enumerate(zip(shape_ids, geometries)):
    # predicate='touches': geometries sharing a boundary but not overlapping
    neighbour_idxs = tree.query(geom, predicate='touches')
    adjacency[sid] = [shape_ids[j] for j in neighbour_idxs if j != i]

total_edges = sum(len(v) for v in adjacency.values())
avg_n = total_edges / len(adjacency) if adjacency else 0
print(f'Adjacency graph built:')
print(f'  Polygons           : {len(adjacency)}')
print(f'  Total directed edges: {total_edges}')
print(f'  Avg neighbours     : {avg_n:.2f}')

Parsing geometries...
Building spatial index...
Finding neighbours...
Adjacency graph built:
  Polygons           : 20850
  Total directed edges: 145142
  Avg neighbours     : 6.96


In [96]:
# Load community taxonomy — valid_types_map still needed for validation and fallback
community_df = pd.read_csv('./csv_input/Info_CSV/Community_matrix.csv')

valid_types_map = {}
for (region, location), grp in community_df.groupby(['Region', 'Type']):
    codes = [str(c).strip() for c in grp['Code'].dropna() if str(c).strip()]
    if not codes:
        continue
    key = f"{region}_{location.replace(' ', '')}"
    valid_types_map[key] = codes

print('Valid type sets per (Region, Location):')
for k, v in valid_types_map.items():
    print(f'  {k:28s}: {v}')


# Build cluster_priors: (Region, Type, Cluster_Name) -> {type_code: probability}
# Restricted to the valid type set for that (Region, Location) and renormalised —
# same pattern the old build_tm_prior used, so a cluster's weight on a type that
# isn't valid there (e.g. GAME/TECH columns, currently all 0) is simply dropped.
cluster_priors = {}
for _, row in cluster_df.iterrows():
    region, location, cluster = row['Region'], row['Type'], row['Cluster_Name']
    valid = valid_types_map.get(f"{region}_{location.replace(' ', '')}", [])
    dist = {t: float(row[t]) for t in _type_cols if t in valid and row[t] > 0}
    total = sum(dist.values())
    if total > 0:
        cluster_priors[(region, location, cluster)] = {t: v / total for t, v in dist.items()}

print(f'\nCluster priors computed : {len(cluster_priors)} (Region, Location, Cluster) combinations')

bad = [(k, round(sum(v.values()), 6)) for k, v in cluster_priors.items()
       if abs(sum(v.values()) - 1.0) > 1e-6]
if bad:
    print(f'WARNING: {len(bad)} cluster priors not summing to 1:')
    for k, s in bad[:5]:
        print(f'  {k}: sum={s}')
else:
    print('All cluster priors normalised to 1.0 ✓')

print('\nSample cluster priors:')
for k, dist in list(cluster_priors.items())[:8]:
    pretty = {t: round(v, 4) for t, v in dist.items()}
    print(f'  {k}: {pretty}')


def get_polygon_prior(region, location, cluster):
    """
    Resolve the prior distribution for one polygon.

    If the polygon has a non-blank 'Clusters' value with a matching row in
    Population_Cluster_Proposed.csv for this (Region, Location), use that
    cluster's type distribution. Otherwise (no cluster, unmatched cluster, or
    a cluster whose valid-restricted mixture summed to 0) fall back to a flat
    uniform prior over the valid types for this (Region, Location).
    """
    valid = valid_types_map.get(f"{region}_{location.replace(' ', '')}", [])
    uniform_fb = {k: 1 / len(valid) for k in valid} if valid else {}

    if pd.notna(cluster) and str(cluster).strip():
        dist = cluster_priors.get((region, location, str(cluster).strip()))
        if dist:
            return dist

    return uniform_fb


print(f'\nget_polygon_prior() defined — resolves per-polygon prior at use time.')

Valid type sets per (Region, Location):
  Capital_InnerCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'S', 'TECH', 'GAME', 'UNI']
  Capital_OuterCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Capital_Rural               : ['A1', 'C1', 'D1', 'D2', 'F1']
  Capital_Suburb              : ['A1', 'B1', 'C1', 'C2', 'E2', 'F1', 'UNI']
  Capital_Town                : ['B2', 'C1', 'C2', 'D2', 'E1', 'F1']
  Highland_InnerCity          : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'S']
  Highland_OuterCity          : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2']
  Highland_Rural              : ['A1', 'C1', 'D1', 'D2', 'F1']
  Highland_Suburb             : ['A1', 'B1', 'C1', 'C2', 'E2', 'F1']
  Highland_Town               : ['B2', 'C1', 'C2', 'D2', 'E1', 'F1']
  Lowland_InnerCity           : ['A1', 'A2', 'B1', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2', 'UNI']
  Lowland

In [97]:
# ── Constants ─────────────────────────────────────────────────────────────────

# Ordered fallback chain — highest to lowest prestige.
# B2 (Mixed New Town) sits between B1 and C1. It only appears in Town priors
# (not in any transition matrix), so it will only ever be sampled for Town
# polygons where it is already valid — the chain covers any cross-location edge case.
FALLBACK_CHAIN = [
    'S', 'A1', 'A2', 'UNI', 'TECH',
    'B1', 'B2', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2'
]

# Types that are skipped during fallback chain walking.
# They are only used as an absolute last resort when no other valid type exists.
_FALLBACK_SKIP = {'UNI', 'TECH'}

# Neighbour-vs-prior blend weight, out of 100. Three distinct stages:
#   1. One-pass seed     : INFLUENCED_WEIGHT_SEED  = 0  (pure prior, 0% neighbour).
#      The "neighbours" visible during this very first, still-forming pass are
#      mostly other in-scope polygons picked moments earlier — not a settled
#      signal worth blending in. Seeding purely from the prior keeps the seed
#      stable and gives the Gibbs sweeps a clean, deterministic-from-prior start.
#   2. Gibbs sweep 1      : INFLUENCED_WEIGHT_START = 30 (30% neighbour / 70% prior)
#   3. Gibbs final sweep  : INFLUENCED_WEIGHT_END   = 50 (50% neighbour / 50% prior)
#      An earlier version ramped this up to 80% and ran 5 sweeps — but every
#      candidate is resampled using OTHER candidates' still-changing values each
#      sweep, so a high, rising neighbour weight let changes cascade (A depends
#      on B, B on C, C back on A) instead of damping out: observed change_rate
#      stayed ~83-87% for all 5 sweeps, never approaching the 1% threshold.
#      Capping the ceiling at 50% and cutting to 3 sweeps (see N_MAX_SWEEPS in
#      the Gibbs cell) keeps that feedback loop weaker.
# compute_likelihood reads the current value of the plain INFLUENCED_WEIGHT
# global each call, so reassigning it between stages is enough — no signature
# change needed.
INFLUENCED_WEIGHT_SEED  = 0
INFLUENCED_WEIGHT_START = 30
INFLUENCED_WEIGHT_END   = 50
INFLUENCED_WEIGHT       = INFLUENCED_WEIGHT_SEED   # used by the one-pass loop

# GAME is a rare/special-case division that will get its own dedicated method
# later. It's excluded from the neighbour-signal component in both directions:
# (1) as a destination column — it isn't a type any polygon is ever assigned
#     (not in valid_types_map / FALLBACK_CHAIN), so it must never receive
#     probability mass in a returned likelihood;
# (2) as a source — polygons in the source data can carry County_Type == 'GAME'
#     as frozen pre-assigned anchors. Rather than let their transition-matrix
#     row (0.6 self-loop / 0.3->TECH / 0.1->B1) partially leak into their
#     neighbours' likelihoods, such neighbours are skipped entirely in
#     compute_likelihood — same treatment as B2 neighbours, which aren't in
#     any transition matrix's index either.
EXCLUDED_NEIGHBOUR_TYPES = {'GAME'}


# ── Helper functions ──────────────────────────────────────────────────────────

def _prior_key(region, location):
    return f"{region}_{location.replace(' ', '')}"


def get_valid_types(region, location):
    """Return the list of valid type codes for a (Region, Location) pair."""
    return valid_types_map.get(_prior_key(region, location), [])


def select_from_distribution(dist):
    """
    Sample one key from a {type: probability} dict using return_ran().
    Maps the raw int [1, 10_000_000] to a uniform [0, 1) value and uses
    the inverse-CDF method to select a bucket.
    """
    types = list(dist.keys())
    probs = list(dist.values())
    total = sum(probs)
    probs = [p / total for p in probs]

    rand_val   = (return_ran() - 1) / (10_000_000 - 1)   # uniform in [0, 1)
    cumulative = 0.0
    for t, p in zip(types, probs):
        cumulative += p
        if rand_val < cumulative:
            return t
    return types[-1]   # floating-point guard


def find_valid_fallback(selected_type, valid_type_list):
    """
    If selected_type is not in valid_type_list, walk FALLBACK_CHAIN to find
    the nearest valid type (downward first, then upward).

    UNI and TECH are skipped in the first pass — they are only accepted as a
    last resort if no other valid type can be found anywhere in the chain.

    For any type not present in FALLBACK_CHAIN (idx = -1), the downward search
    starts from the beginning of the chain (range(0, len)).
    """
    if not valid_type_list:
        return None
    if selected_type in valid_type_list:
        return selected_type

    try:
        idx = FALLBACK_CHAIN.index(selected_type)
    except ValueError:
        idx = -1   # type absent from chain — search the whole chain downward

    # Pass 1: walk the chain excluding UNI and TECH
    for i in range(idx + 1, len(FALLBACK_CHAIN)):
        if FALLBACK_CHAIN[i] in valid_type_list and FALLBACK_CHAIN[i] not in _FALLBACK_SKIP:
            return FALLBACK_CHAIN[i]
    for i in range(idx - 1, -1, -1):
        if FALLBACK_CHAIN[i] in valid_type_list and FALLBACK_CHAIN[i] not in _FALLBACK_SKIP:
            return FALLBACK_CHAIN[i]

    # Pass 2: nothing found without UNI/TECH — allow them as a last resort
    for i in range(idx + 1, len(FALLBACK_CHAIN)):
        if FALLBACK_CHAIN[i] in valid_type_list:
            return FALLBACK_CHAIN[i]
    for i in range(idx - 1, -1, -1):
        if FALLBACK_CHAIN[i] in valid_type_list:
            return FALLBACK_CHAIN[i]

    return valid_type_list[0]   # absolute last resort


def compute_likelihood(region, location, cluster, classified_neighbours):
    """
    Blend neighbour transition signal with the polygon's cluster-weighted prior.

    Parameters
    ----------
    region, location : str
        Attributes of the target polygon.
    cluster : str or NaN
        The polygon's 'Clusters' value, resolved to a prior via get_polygon_prior
        (cluster-specific distribution if matched, else flat uniform fallback).
    classified_neighbours : list of (type_code, location_str)
        Neighbours that already have a County_Type.
        Each is looked up in its *own* location's transition matrix.

    Returns
    -------
    dict {type_code: probability}  — normalised to sum = 1

    Notes
    -----
    - Reads the module-level INFLUENCED_WEIGHT at call time, so callers can
      change the prior/neighbour balance between stages just by reassigning
      that global — see INFLUENCED_WEIGHT_SEED/START/END above.
    - Neighbours whose type is in EXCLUDED_NEIGHBOUR_TYPES (currently just GAME)
      are skipped entirely — same treatment as B2 neighbours (not in TM index).
      They don't contribute to neighbour_sum and don't count toward `count`.
    - B2 is not a column in any transition matrix, so it receives 0 from the
      neighbour component. Its prior weight is still preserved for Town
      polygons via the prior term.
    """
    prior = get_polygon_prior(region, location, cluster)

    if not classified_neighbours:
        return dict(prior)

    # Collect all type columns present across all matrices, minus excluded ones
    all_type_cols = set()
    for tm in transition_matrices.values():
        all_type_cols.update(tm.columns.tolist())
    all_type_cols -= EXCLUDED_NEIGHBOUR_TYPES

    # Sum transition rows from every classified neighbour
    neighbour_sum = {t: 0.0 for t in all_type_cols}
    count = 0

    for n_type, n_location in classified_neighbours:
        if n_type in EXCLUDED_NEIGHBOUR_TYPES:
            continue   # GAME-typed neighbours skipped entirely — no dedicated method yet
        tm = transition_matrices.get(n_location)
        if tm is None or n_type not in tm.index:
            continue   # B2 neighbours (not in TM index) are skipped here
        row = tm.loc[n_type]
        for t in all_type_cols:
            if t in row.index:
                neighbour_sum[t] += float(row[t])
        count += 1

    if count == 0:
        return dict(prior)   # all neighbours were untraceable — fall back to prior

    neighbour_avg = {t: neighbour_sum[t] / count for t in all_type_cols}

    # Blend: influenced_weight * neighbour_avg + (100 - influenced_weight) * prior
    # Union includes B2 for Town polygons (from prior) even though TMs lack it
    all_keys = set(prior.keys()) | all_type_cols
    blended  = {}
    for t in all_keys:
        n_prob = neighbour_avg.get(t, 0.0)   # 0 for B2 (not in TMs)
        p_prob = prior.get(t, 0.0)
        blended[t] = INFLUENCED_WEIGHT * n_prob + (100 - INFLUENCED_WEIGHT) * p_prob

    total = sum(blended.values())
    if total == 0:
        return dict(prior)

    return {t: v / total for t, v in blended.items() if v > 0}


print('Helper functions defined.')
print(f'FALLBACK_CHAIN           : {FALLBACK_CHAIN}')
print(f'FALLBACK_SKIP            : {_FALLBACK_SKIP}')
print(f'INFLUENCED_WEIGHT_SEED   : {INFLUENCED_WEIGHT_SEED}')
print(f'INFLUENCED_WEIGHT_START  : {INFLUENCED_WEIGHT_START}')
print(f'INFLUENCED_WEIGHT_END    : {INFLUENCED_WEIGHT_END}')
print(f'EXCLUDED_NEIGHBOUR_TYPES : {EXCLUDED_NEIGHBOUR_TYPES}')


Helper functions defined.
FALLBACK_CHAIN           : ['S', 'A1', 'A2', 'UNI', 'TECH', 'B1', 'B2', 'C1', 'C2', 'D2', 'D1', 'E1', 'E2', 'F1', 'F2']
FALLBACK_SKIP            : {'UNI', 'TECH'}
INFLUENCED_WEIGHT_SEED   : 0
INFLUENCED_WEIGHT_START  : 30
INFLUENCED_WEIGHT_END    : 50
EXCLUDED_NEIGHBOUR_TYPES : {'GAME'}


In [98]:
# ── O(1) lookup structures ────────────────────────────────────────────────────
_idx     = work_df.set_index('Shape_ID')
ct_state = _idx['County_Type'].to_dict()
meta     = _idx[['Region', 'Location', 'County', 'Clusters']].to_dict('index')

# Step 1: build unchecked_list — restricted to polygons that actually have a
# Clusters value. At this stage we only classify what the cluster data covers;
# blank-County_Type polygons with no Clusters value are left untouched (neglected)
# rather than falling back to a flat prior — that comes in a later pass.
# pandas already parses a blank CSV cell as real NaN (verified: 0 empty strings,
# 19,886 NaN, 964 notna in Clusters), so pd.notna() alone is the correct check.
blank_ids   = [sid for sid, ct in ct_state.items() if pd.isna(ct) or str(ct).strip() == '']
has_cluster = lambda sid: pd.notna(meta[sid]['Clusters'])

unchecked_list = [sid for sid in blank_ids if has_cluster(sid)]
neglected_list = [sid for sid in blank_ids if not has_cluster(sid)]

print(f'Blank County_Type polygons overall : {len(blank_ids)}')
print(f'  With a Clusters value (in scope) : {len(unchecked_list)}')
print(f'  Without a Clusters value (skipped, left blank this pass): {len(neglected_list)}')
print(f'Already assigned                   : {len(ct_state) - len(blank_ids)}')

# ── Pre-fetch all random numbers in one CSV read ──────────────────────────────
# Each iteration uses at most 3 numbers (step-2 pick, step-6 sample, rare fallback).
# return_ran_array(n) does exactly ONE read + ONE write of randomno.csv for all n ints.
# Everything after this is fully offline — no further file I/O.

_POOL_SIZE  = len(unchecked_list) * 3
_rand_list  = return_ran_array(_POOL_SIZE)
_rand_iter  = iter(_rand_list)

print(f'Random pool        : {_POOL_SIZE} integers pre-fetched (1 CSV read/write total)')

# Snapshot of the unassigned IDs — used by the Gibbs sweep cell.
# Captured here so it is available even after unchecked_list is fully consumed.
# Pre-assigned polygons, and neglected no-cluster polygons, are excluded by
# construction: they were never in unchecked_list and won't appear here.
gibbs_candidates = list(unchecked_list)
print(f'gibbs_candidates   : {len(gibbs_candidates)} polygons (frozen anchors + no-cluster polygons excluded)')

# Rebind return_ran to draw from the in-memory list so select_from_distribution
# also uses the pool without any code changes in that function.
def return_ran():
    return next(_rand_iter)

# ── Main assignment loop (fully offline) ──────────────────────────────────────
iteration = 0
log_every = max(1, len(unchecked_list) // 10)

while unchecked_list:
    iteration += 1

    # Step 2: random pick from unchecked_list
    rand_no     = return_ran()
    pick_idx    = (rand_no - 1) % len(unchecked_list)
    selected_id = unchecked_list[pick_idx]

    region   = meta[selected_id]['Region']
    location = meta[selected_id]['Location']

    # Step 4: gather classified neighbours from adjacency graph
    classified_neighbours = []
    for n_id in adjacency.get(selected_id, []):
        n_ct = ct_state.get(n_id)
        if n_ct and not pd.isna(n_ct) and str(n_ct).strip():
            classified_neighbours.append((str(n_ct).strip(), meta[n_id]['Location']))

    # Steps 5a / 5b: compute blended likelihood
    cluster    = meta[selected_id]['Clusters']
    likelihood = compute_likelihood(region, location, cluster, classified_neighbours)

    # Step 6: sample type from likelihood
    # select_from_distribution calls return_ran() — rebound above, draws from pool
    if likelihood:
        selected_type = select_from_distribution(likelihood)
    else:
        valid = get_valid_types(region, location)
        selected_type = valid[(return_ran() - 1) % len(valid)] if valid else None

    # Step 6 (cont.): validate; walk fallback chain if type not valid for location
    if selected_type:
        valid_for_loc = get_valid_types(region, location)
        if selected_type not in valid_for_loc:
            selected_type = find_valid_fallback(selected_type, valid_for_loc)

    # Step 7: assign
    if selected_type:
        ct_state[selected_id] = selected_type

    # Step 8: remove from unchecked_list and continue
    unchecked_list.pop(pick_idx)

    if iteration % log_every == 0 or not unchecked_list:
        print(f'  [{iteration:>5d}] remaining: {len(unchecked_list)}')

# Write one-pass results back to work_df (intermediate snapshot — Gibbs will refine)
work_df['County_Type'] = work_df['Shape_ID'].map(ct_state)

print(f'\nOne-pass complete — {iteration} polygons assigned.')
print(f'Unassigned remaining (incl. {len(neglected_list)} no-cluster polygons neglected this pass): {work_df["County_Type"].isna().sum()}')
print('\nCounty_Type distribution:')
print(work_df['County_Type'].value_counts())

Blank County_Type polygons overall : 5909
  With a Clusters value (in scope) : 5870
  Without a Clusters value (skipped, left blank this pass): 39
Already assigned                   : 14941
Random pool        : 17610 integers pre-fetched (1 CSV read/write total)
gibbs_candidates   : 5870 polygons (frozen anchors + no-cluster polygons excluded)
  [  587] remaining: 5283
  [ 1174] remaining: 4696
  [ 1761] remaining: 4109
  [ 2348] remaining: 3522
  [ 2935] remaining: 2935
  [ 3522] remaining: 2348
  [ 4109] remaining: 1761
  [ 4696] remaining: 1174
  [ 5283] remaining: 587
  [ 5870] remaining: 0

One-pass complete — 5870 polygons assigned.
Unassigned remaining (incl. 39 no-cluster polygons neglected this pass): 39

County_Type distribution:
County_Type
D2      2699
D1      2591
F1      2532
C1      2352
E2      2111
C2      1805
B1      1632
F2      1499
E1      1279
A1       658
A2       626
TECH     485
UNI      270
GAME     109
B2        85
S         78
Name: count, dtype: int64


In [99]:
import random
from itertools import cycle

# ── Gibbs sweep configuration ─────────────────────────────────────────────────
CONVERGENCE_THRESHOLD = 0.01   # stop when < 1 % of polygons change type in a sweep
N_MAX_SWEEPS          = 1     # cut from 5 — with a rising neighbour weight, 5 sweeps
                               # never converged (change_rate stayed ~83-87% every
                               # sweep). Fewer sweeps at a lower, capped weight keeps
                               # the candidate-resamples-candidate feedback loop weaker.

# ── Pre-fetch random pool (one CSV read/write for all Gibbs sweeps) ───────────
# Pool = 10 × number of Gibbs candidates.
# Each sweep consumes exactly len(gibbs_candidates) numbers (one per polygon for
# select_from_distribution). The pool covers ~10 sweeps without cycling; if
# convergence takes longer, itertools.cycle wraps silently.
# Cycling still retains effective randomness: the same raw value r maps to a
# different type each time because the likelihood distribution has changed.
_GIBBS_POOL_SIZE = 10 * len(gibbs_candidates)
_gibbs_pool      = return_ran_array(_GIBBS_POOL_SIZE)
_rand_iter       = cycle(_gibbs_pool)

# Rebind return_ran so select_from_distribution draws from the Gibbs pool
def return_ran():
    return next(_rand_iter)

print(f'Gibbs sweep setup')
print(f'  Candidates         : {len(gibbs_candidates)}')
print(f'  Random pool        : {_GIBBS_POOL_SIZE} integers (10x, cycles silently if needed)')
print(f'  Convergence thresh : < {CONVERGENCE_THRESHOLD*100:.0f}% changes per sweep')
print(f'  Max sweeps         : {N_MAX_SWEEPS}')
print(f'  Weight schedule    : {INFLUENCED_WEIGHT_START} -> {INFLUENCED_WEIGHT_END} (neighbour %, linear over sweeps; one-pass seed used {INFLUENCED_WEIGHT_SEED}%)')
print()

# ── Gibbs sweeps ──────────────────────────────────────────────────────────────
for sweep in range(1, N_MAX_SWEEPS + 1):

    # Ramp INFLUENCED_WEIGHT from START (sweep 1) to END (final sweep), linearly.
    # compute_likelihood reads this global at call time, so reassigning it here
    # is all that's needed — every call this sweep uses the new value.
    INFLUENCED_WEIGHT = round(
        INFLUENCED_WEIGHT_START +
        (INFLUENCED_WEIGHT_END - INFLUENCED_WEIGHT_START) * (sweep - 1) / max(N_MAX_SWEEPS - 1, 1)
    )

    # Random sweep order — uses Python's own RNG, independent of the pool
    random.shuffle(gibbs_candidates)

    changes = 0

    for selected_id in gibbs_candidates:
        region   = meta[selected_id]['Region']
        location = meta[selected_id]['Location']
        cluster  = meta[selected_id]['Clusters']

        # Collect ALL current neighbours — all polygons are now assigned
        classified_neighbours = [
            (str(ct_state[n_id]).strip(), meta[n_id]['Location'])
            for n_id in adjacency.get(selected_id, [])
            if ct_state.get(n_id) and not pd.isna(ct_state[n_id])
            and str(ct_state[n_id]).strip()
        ]

        # Full conditional: likelihood from complete neighbourhood
        likelihood = compute_likelihood(region, location, cluster, classified_neighbours)

        if likelihood:
            new_type = select_from_distribution(likelihood)
        else:
            valid    = get_valid_types(region, location)
            new_type = valid[(return_ran() - 1) % len(valid)] if valid else None

        # Validate; walk fallback chain if needed
        if new_type:
            valid_for_loc = get_valid_types(region, location)
            if new_type not in valid_for_loc:
                new_type = find_valid_fallback(new_type, valid_for_loc)

        # Record change and update shared state
        if new_type and new_type != ct_state[selected_id]:
            changes += 1
            ct_state[selected_id] = new_type

    change_rate = changes / len(gibbs_candidates)
    print(f'  Sweep {sweep:>3d} : {changes:>5d} changes  ({change_rate*100:5.2f}%)  INFLUENCED_WEIGHT={INFLUENCED_WEIGHT}')

    if change_rate < CONVERGENCE_THRESHOLD:
        print(f'\nConverged after {sweep} sweep(s).')
        break

else:
    print(f'\nReached max sweeps ({N_MAX_SWEEPS}) — did not converge below threshold.')

# ── Write final Gibbs-refined results back to work_df ─────────────────────────
work_df['County_Type'] = work_df['Shape_ID'].map(ct_state)

print(f'\nFinal unassigned : {work_df["County_Type"].isna().sum()}')
print('\nFinal County_Type distribution after Gibbs:')
print(work_df['County_Type'].value_counts())

Gibbs sweep setup
  Candidates         : 5870
  Random pool        : 58700 integers (10x, cycles silently if needed)
  Convergence thresh : < 1% changes per sweep
  Max sweeps         : 1
  Weight schedule    : 30 -> 50 (neighbour %, linear over sweeps; one-pass seed used 0%)

  Sweep   1 :  4484 changes  (76.39%)  INFLUENCED_WEIGHT=30

Reached max sweeps (1) — did not converge below threshold.

Final unassigned : 39

Final County_Type distribution after Gibbs:
County_Type
D2      2722
D1      2612
F1      2423
C1      2293
E2      2180
C2      1744
B1      1648
F2      1515
E1      1341
A1       665
A2       642
TECH     485
UNI      273
GAME     109
B2        81
S         78
Name: count, dtype: int64


In [ ]:
work_df.to_csv('./csv_input/test6_assigned_county_types_updated.csv', index=False)

: 

In [ ]:
# ── Print work_df overview ────────────────────────────────────────────────────
print('work_df (first 10 rows):')
print(work_df[['County', 'Shape_ID', 'Location', 'County_Type']].head(10).to_string())
print(f'\nShape: {work_df.shape}')
print(f'\nCounty_Type value counts:\n{work_df["County_Type"].value_counts()}')

# ── Column order: follow FALLBACK_CHAIN hierarchy (left = highest prestige) ───
# Only keep types that actually appear in the data
col_order = [t for t in FALLBACK_CHAIN if t in work_df['County_Type'].values]

# ── group_county_work_df ──────────────────────────────────────────────────────
# Rows = County, Columns = County_Type (hierarchy order), Values = % within county

county_counts = (
    work_df
    .groupby(['County', 'County_Type'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=col_order, fill_value=0)
)

group_county_work_df = (
    county_counts
    .div(county_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

print('\ngroup_county_work_df  (County_Type % per County, left→right = high→low prestige):')
print(group_county_work_df.to_string())

# ── group_district_work_df ────────────────────────────────────────────────────
# County code split into letter prefix + numeric suffix  e.g. "NC1" → "NC" | "1"
# Group only by the letter prefix (district)

work_df['_district'] = work_df['County'].str.extract(r'^([A-Za-z]+)')

district_counts = (
    work_df
    .groupby(['_district', 'County_Type'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=col_order, fill_value=0)
)

group_district_work_df = (
    district_counts
    .div(district_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)
group_district_work_df.index.name = 'District'

print('\ngroup_district_work_df  (County_Type % per District, left→right = high→low prestige):')
print(group_district_work_df.to_string())

# Clean up temporary column
work_df.drop(columns=['_district'], inplace=True)

In [ ]:
group_district_work_df.to_csv('inspect_group_district_work_df.csv')
group_county_work_df.to_csv('inspect_group_county_work_df.csv')

In [ ]:
group_district_work_df